# Question 4 - GAN Generator vs Discriminator Loss (MNIST)
- Discriminator: $L_D = BCE(D(x_{real}), 1) + BCE(D(G(z)), 0)$
- Non-saturating generator: $L_G = BCE(D(G(z)), 1)$

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import matplotlib.pyplot as plt

In [ ]:
bce = keras.losses.BinaryCrossentropy()

def discriminator_loss(d_real, d_fake):
    return bce(tf.ones_like(d_real), d_real) + bce(tf.zeros_like(d_fake), d_fake)

def generator_loss(d_fake):          # non-saturating
    return bce(tf.ones_like(d_fake), d_fake)

In [ ]:
# Load + preprocess MNIST to [-1, 1]
(x_train, _), _ = keras.datasets.mnist.load_data()
x_train = (x_train.astype("float32") - 127.5) / 127.5
x_train = x_train.reshape(-1, 28, 28, 1)
dataset = tf.data.Dataset.from_tensor_slices(x_train).shuffle(10000).batch(128)
LATENT = 100

In [ ]:
class GAN:
    def __init__(self):
        self.G = keras.Sequential([layers.Input((LATENT,)),
                                   layers.Dense(256), layers.LeakyReLU(0.2),
                                   layers.Dense(512), layers.LeakyReLU(0.2),
                                   layers.Dense(784, activation='tanh'),
                                   layers.Reshape((28, 28, 1))])
        self.D = keras.Sequential([layers.Input((28, 28, 1)), layers.Flatten(),
                                   layers.Dense(512), layers.LeakyReLU(0.2),
                                   layers.Dense(256), layers.LeakyReLU(0.2),
                                   layers.Dense(1, activation='sigmoid')])
        self.g_opt = keras.optimizers.Adam(2e-4, beta_1=0.5)
        self.d_opt = keras.optimizers.Adam(2e-4, beta_1=0.5)

    @tf.function
    def train_step(self, real):
        z = tf.random.normal((tf.shape(real)[0], LATENT))
        with tf.GradientTape() as d_tape, tf.GradientTape() as g_tape:
            fake = self.G(z, training=True)
            d_real = self.D(real, training=True)
            d_fake = self.D(fake, training=True)
            d_loss = discriminator_loss(d_real, d_fake)
            g_loss = generator_loss(d_fake)
        self.d_opt.apply_gradients(zip(d_tape.gradient(d_loss, self.D.trainable_variables), self.D.trainable_variables))
        self.g_opt.apply_gradients(zip(g_tape.gradient(g_loss, self.G.trainable_variables), self.G.trainable_variables))
        return d_loss, g_loss

gan = GAN()
for epoch in range(3):
    for real in dataset:
        d_loss, g_loss = gan.train_step(real)
    print(f"epoch {epoch+1}: D loss={d_loss:.3f}  G loss={g_loss:.3f}")

In [ ]:
imgs = gan.G(tf.random.normal((16, LATENT))).numpy()
plt.figure(figsize=(4, 4))
for i in range(16):
    plt.subplot(4, 4, i + 1); plt.imshow(imgs[i, :, :, 0], cmap='gray'); plt.axis('off')
plt.show()

In [ ]:
# Gradient comparison: saturating log(1-D) vs non-saturating -log(D)
d = np.linspace(0.001, 0.999, 200)
plt.plot(d, np.abs(-1 / (1 - d)), label='|d/dD log(1-D)|  (saturating)')
plt.plot(d, np.abs(-1 / d), label='|d/dD -log D|  (non-saturating)')
plt.yscale('log'); plt.xlabel('D(G(z))'); plt.ylabel('gradient magnitude'); plt.legend(); plt.show()

### 2. Why is $L_G = BCE(D(G(z)), 1)$ preferred early in training?
Early on the generator is poor, so $D(G(z)) \approx 0$. At that point $\log(1 - D(G(z)))$ is flat (gradient about $-1$ in D, and close to zero with respect to the logits after the sigmoid), so the **gradients vanish (saturate)** and G barely learns.

$-\log D(G(z))$ has a **large gradient exactly when $D(G(z))$ is near 0**, so G gets a strong signal. Both losses have the same fixed point, but the non-saturating one learns much faster at the start.